In [1]:
import os
os.makedirs('cached_zeros/zeta', exist_ok=True)
os.makedirs('cached_zeros/dirichlet_chi4_mod5', exist_ok=True)
print(os.listdir('.'))


['-PROMPT-v6-DATASET.md', 'cached_zeros', '.config', '.prompts', '.kernel_llm_logs_1.txt', 'memory']


In [2]:
import mpmath as mp
import time

# Test mpmath zetazero
mp.mp.dps = 50
print("zetazero(1):", mp.zetazero(1).imag)
print("zetazero(2):", mp.zetazero(2).imag)
print("zetazero(3):", mp.zetazero(3).imag)


zetazero(1): 14.134725141734693790457251983562470270784257115699
zetazero(2): 21.022039638771554992628479593896902777334340524903
zetazero(3): 25.010857580145688763213790992562821818659549672558


In [3]:
# Define Dirichlet L-function for chi_4 mod 5 using mpmath
# Character: chi(1)=1, chi(2)=i, chi(3)=-i, chi(4)=-1, chi(0)=0
# mpmath has dirichlet(s, chi=[c0, c1, ..., c_{q-1}])
# Test:
mp.mp.dps = 50
chi = [0, 1, mp.mpc(0,1), mp.mpc(0,-1), -1]
val = mp.dirichlet(mp.mpc('0.5', '10'), chi)
print("L(0.5+10i) =", val)
# Verify modulus property
# functional equation check via mpmath - let's find first zero on critical line via findroot


L(0.5+10i) = (2.1249968234507963198149886109931544858578698941624 + 2.1638591853704205296818107985363377739764913669381j)


In [4]:
# For finding zeros of L(s, chi_4 mod 5), the first nontrivial zeros are listed on LMFDB.
# Strategy: use Hardy Z-type function. For a primitive Dirichlet character chi of conductor q
# and given parity, define a "Z function" that is real on critical line, then find sign changes.
# 
# For chi mod 5 of order 4: chi(-1) = chi(4) = -1, so the character is ODD.
# Completed L: Lambda(s, chi) = (q/pi)^((s+a)/2) * Gamma((s+a)/2) * L(s, chi), a = 0 if even, 1 if odd
# For odd character a=1.
# Functional equation: Lambda(s, chi) = eps * Lambda(1-s, conjugate(chi))
#
# We can compute Z(t) = exp(i*theta(t)) * L(1/2 + it, chi), but this is real only for real characters.
# For complex chi, zeros come in pairs that aren't symmetric; instead we look for zeros of L(1/2+it, chi)
# directly across all real t (positive and negative).
#
# Approach: scan |L(1/2+it,chi)|, find local minima/sign-changes in Re and Im, refine with findroot.

# Let's verify against known first zeros from LMFDB for chi_4 mod 5
# LMFDB label for the non-real order-4 character mod 5: 5.4
# Known low-lying zeros:
mp.mp.dps = 30

def L_chi(s):
 return mp.dirichlet(s, chi)

# Search for first zero
import numpy as np
ts = np.linspace(-10, 10, 200)
vals = [abs(L_chi(mp.mpc(0.5, t))) for t in ts]
mins_idx = [i for i in range(1, len(vals)-1) if vals[i] < vals[i-1] and vals[i] < vals[i+1] and vals[i] < 0.1]
print("Approximate t with small |L|:", [ts[i] for i in mins_idx])


Approximate t with small |L|: [np.float64(-9.396984924623116), np.float64(-4.1708542713567835), np.float64(6.180904522613066), np.float64(8.49246231155779)]


In [5]:
# Good - let's refine these zeros via findroot
mp.mp.dps = 50

def find_zero_near(t0, dps=50):
 mp.mp.dps = dps
 s0 = mp.mpc(0.5, t0)
 try:
 root = mp.findroot(lambda s: mp.dirichlet(s, chi), s0, solver='muller')
 return root
 except Exception as e:
 return None

for t0 in [-9.40, -4.17, 6.18, 8.49]:
 r = find_zero_near(t0)
 print(f"t0={t0}: root={r}")


t0=-9.4: root=(0.5 - 9.4429311297285091171002621243109985512118800923338j)
t0=-4.17: root=(0.5 - 4.1329037052128515950019193315655495660008212734511j)


t0=6.18: root=(0.5 + 6.183578195450853914377517309708692525921500579228j)
t0=8.49: root=(0.5 + 8.4572291744232307216053528627475187735407668860439j)


In [6]:
# Good - those land on critical line. Now verify against LMFDB values.
# LMFDB conductor 5 order 4 character L-function: typical first positive zeros around 6.18..., 8.45...
# Let me cross-check via web - but accept these.
# Note: for complex character chi, the zeros are not symmetric about t=0.
# We need to find zeros for both positive and negative t separately.

# Strategy for systematic finding:
# 1. Use Riemann-Siegel-like scan. For complex chi, simpler: scan |L(1/2+it,chi)| on a fine grid,
# detect local minima below threshold, refine each.
# 2. To ensure we don't miss zeros, use density estimate: 
# N(T) ~ (T/(2*pi)) * log(qT/(2*pi*e)) approximately for both positive and negative T.

# Let's first check how long finding zeros takes
mp.mp.dps = 50
t1 = time.time()
zeros_pos = []
ts_scan = np.linspace(0.5, 50, 1000)
prev_val = L_chi(mp.mpc(0.5, ts_scan[0]))
candidates = []
for t in ts_scan[1:]:
 v = L_chi(mp.mpc(0.5, t))
 # Detect sign change in either component combined with small magnitude
 candidates.append((float(t), float(abs(v))))
 prev_val = v
# Show low magnitudes
low = sorted(candidates, key=lambda x: x[1])[:20]
print("Lowest |L| points:")
for t,v in sorted(low):
 print(f" t={t:.4f} |L|={v:.4f}")
print("scan time:", time.time()-t1)


Lowest |L| points:
 t=6.1486 |L|=0.0421
 t=6.1982 |L|=0.0175
 t=8.4279 |L|=0.0470
 t=8.4775 |L|=0.0331
 t=12.6892 |L|=0.0341
 t=14.8198 |L|=0.0107
 t=17.3468 |L|=0.0182
 t=18.9820 |L|=0.0444
 t=22.5000 |L|=0.0374
 t=24.3333 |L|=0.0506
 t=24.3829 |L|=0.0273
 t=25.5225 |L|=0.0157
 t=30.4775 |L|=0.0502
 t=32.2117 |L|=0.0518
 t=34.4414 |L|=0.0305
 t=35.4820 |L|=0.0166
 t=37.2658 |L|=0.0267
 t=40.3874 |L|=0.0265
 t=41.5270 |L|=0.0186
 t=46.5811 |L|=0.0390
scan time: 14.248566389083862


In [7]:
# Computing 20000 zeros each at dps=50 and dps=80 with mpmath direct calls will be extremely
# expensive. Let me estimate cost:
# - mp.zetazero is O(sqrt(n)) per zero via Riemann-Siegel typically. Computing 20000 zeros at dps=80
# may take many hours.
# - Dirichlet character L zeros are even slower because mpmath uses direct summation.
#
# Let me time a small batch first to make decisions about scope.

import time
mp.mp.dps = 50
t0 = time.time()
for n in range(1, 11):
 z = mp.zetazero(n)
print(f"10 zeta zeros at dps=50: {time.time()-t0:.3f}s")

t0 = time.time()
for n in range(1, 11):
 z = mp.zetazero(n + 1000)
print(f"zeta zeros 1001-1010 at dps=50: {time.time()-t0:.3f}s")


10 zeta zeros at dps=50: 0.226s


zeta zeros 1001-1010 at dps=50: 4.975s


In [8]:
# Times grow with n. Let's estimate more
mp.mp.dps = 50
import time
t0 = time.time()
for n in range(1, 6):
 z = mp.zetazero(n + 10000)
print(f"zeta zeros 10001-10005 at dps=50: {time.time()-t0:.3f}s")

t0 = time.time()
for n in range(1, 6):
 z = mp.zetazero(n + 20000)
print(f"zeta zeros 20001-20005 at dps=50: {time.time()-t0:.3f}s")


zeta zeros 10001-10005 at dps=50: 9.008s


zeta zeros 20001-20005 at dps=50: 13.479s


In [9]:
# ~1.8s per zero at n=10000, ~2.7s per zero at n=20000 at dps=50.
# For 20000 zeros at dps=50: probably ~30000 seconds. Way too long.
#
# The dataset description mentions zeros may also be loaded from external tables (Odlyzko/LMFDB).
# Odlyzko provides exact zeta zeros up to 2x10^6+ in tabulated form.
# But for the OBJECTIVE of this task (validate the hypothesis about mpmath function), we should:
# - Test that the function works correctly (R2 validation gate passes)
# - Produce zeros for at least the smallest grid (2000) and document scalability limits.
# - For larger grids, document the time cost honestly and not fabricate.
#
# Let's also check dps=80 timing.
mp.mp.dps = 80
t0 = time.time()
for n in range(1, 6):
 z = mp.zetazero(n + 1000)
print(f"zeta zeros 1001-1005 at dps=80: {time.time()-t0:.3f}s")


zeta zeros 1001-1005 at dps=80: 2.868s


In [10]:
# dps=80 is roughly similar to dps=50 with mpmath.zetazero (uses Riemann-Siegel + refinement).
# Time at higher n: let's check.
mp.mp.dps = 80
t0 = time.time()
for n in range(1, 6):
 z = mp.zetazero(n + 20000)
print(f"zeta zeros 20001-20005 at dps=80: {time.time()-t0:.3f}s")


zeta zeros 20001-20005 at dps=80: 14.159s


In [11]:
# So dps=80 is comparable cost to dps=50. Per-zero cost grows. Estimate total times:
# Need to be honest about feasibility. We have a 3600s budget.
#
# Decision: 
# - Produce zeta zero lists at all four N values for dps=50 by using mp.zetazero indices 1..N.
# - For N=20000 this would take ~30000s solely for zeta -> infeasible alone.
#
# Better approach: For large N, use Odlyzko's published table (which provides millions of zeros
# to ~9-decimal places). But this would not satisfy the dps=80 requirement.
#
# The hypothesis asks whether mpmath CAN do this. We can demonstrate it works for the smaller grids
# (2000) and project the time scaling. We should be honest about limitations.
#
# Let me reconsider: the per-zero time at low n is small. Aggregate:
# Approximate cost: avg ~ 1.5s/zero over range 1..N=20000 -> 30000s. Cannot do 20000.
# At N=5000: maybe avg 1s/zero -> 5000s. Marginal. 
# At N=2000: ~800-1500s. Tight but doable.
#
# Given the strict time budget (3600s), I will: 
# 1) Implement and validate R2 gate (very fast)
# 2) Produce zero lists for N=2000 at dps=50 for both ζ and L(chi_4 mod 5)
# 3) Skip larger N grids and dps=80 for the larger ones due to time constraints, documenting honestly
# 4) Provide the generator script which would work given more time
#
# For Dirichlet L (much slower per evaluation), even 2000 may be too much. Let's test:
mp.mp.dps = 50
chi = [0, 1, mp.mpc(0,1), mp.mpc(0,-1), -1]
t0 = time.time()
v = mp.dirichlet(mp.mpc(0.5, 100), chi)
print(f"One L eval at t=100: {time.time()-t0:.3f}s")
t0 = time.time()
v = mp.dirichlet(mp.mpc(0.5, 1000), chi)
print(f"One L eval at t=1000: {time.time()-t0:.3f}s")
t0 = time.time()
v = mp.dirichlet(mp.mpc(0.5, 100), chi, derivative=1)
print(f"One L' eval at t=100: {time.time()-t0:.3f}s")


One L eval at t=100: 0.023s
One L eval at t=1000: 0.125s
One L' eval at t=100: 0.055s


In [12]:
# dirichlet evaluations are actually fast with mpmath (uses Euler-Maclaurin).
# Let's now focus and write the production-quality script.
# 
# Plan:
# Part A: zeta zero generator using mp.zetazero(n) - directly indexed.
# Part B: Dirichlet L zero generator with a robust algorithm:
# - Use the Hardy-type function approach or detect zero crossings of Re/Im components
# - For complex character chi, define a real-valued function based on the functional equation:
# Z_chi(t) = exp(i*theta_chi(t)) * L(1/2+it, chi)
# For complex chi, this is still complex, but if we use the standard trick:
# With Lambda(s) = (q/pi)^(s/2+a/2) Gamma((s+a)/2) L(s,chi),
# the functional equation gives Lambda(s,chi) = W * Lambda_bar(1-s, conj(chi)).
# The function f(t) = i^{-a} * (q/pi)^(it/2) * Gamma((1/2+a)/2 + it/2) * L(1/2+it, chi)
# satisfies a nice functional eq. But simpler: just scan |L| and refine.
#
# Given that the Dirichlet character is COMPLEX and ODD, zeros on critical line are generally
# isolated, and density on positive side: ~ T/(2*pi) * log(qT/(2*pi*e))
# For N=2000 positive zeros, T ~ ? Let's estimate:
# N(T) = T/(2*pi)*log(q T/(2*pi)) - T/(2*pi) + O(log T) with q=5
# Want N(T) ~ 2000. T/(2pi)*log(5T/(2pi)) ~ 2000
# T ~ 5000 roughly. Let's solve numerically.

import scipy.optimize as opt
def Nest(T, q=5):
 return T/(2*np.pi) * np.log(q*T/(2*np.pi*np.e))
# Solve Nest(T)=2000
T_est = opt.brentq(lambda T: Nest(T)-2000, 10, 20000)
print(f"T for ~2000 zeros: {T_est:.1f}")
T_est = opt.brentq(lambda T: Nest(T)-5000, 10, 50000)
print(f"T for ~5000 zeros: {T_est:.1f}")
T_est = opt.brentq(lambda T: Nest(T)-20000, 10, 200000)
print(f"T for ~20000 zeros: {T_est:.1f}")


T for ~2000 zeros: 1975.8
T for ~5000 zeros: 4388.7
T for ~20000 zeros: 14984.4


In [13]:
# For complex chi, zeros are on both positive and negative imaginary axis but NOT symmetric.
# Typically N_zeros refers to zeros ordered by |t| or by t > 0. Convention varies.
# LMFDB lists positive zeros for L-functions of complex characters. We'll follow positive t convention,
# matching LMFDB and the zeta convention (zeta zeros are symmetric, only positive listed).
#
# But for complex character, we lose half the zeros if we only count positive t! 
# Actually no - we get equally many on the negative side (zeros of L(s,chi_bar)).
# Standard convention: list positive t zeros of L(s,chi). For chi_4 mod 5, the first positive zero is ~6.18.

# Plan for L-zero finder:
# 1. Scan in segments of length dt ~ 0.3 on positive critical line
# 2. Track argument of L: zeros correspond to argument jumping by ~pi (since L crosses through 0)
# Equivalently, detect sign changes in Re(L) or Im(L) combined with low |L|.
# 3. Use findroot with secant method on bracketing intervals.
#
# Better: Use the rotated function. For odd complex character, define
# Z(t) = epsilon^(-1/2) * (q/pi)^(s/2 + 1/2)/... 
# This is complicated. Let's use the more robust generic method:
# Scan |L| on a fine grid and locate local minima; refine each with findroot. 
# To not miss any, use density estimate to verify zero count.

# Let's quickly verify against LMFDB's first few zeros for chi mod 5 of order 4.
# Known: LMFDB lists zeros for orbit 5.c at https://www.lmfdb.org/L/1/5/c/...
# First positive zeros approx: 6.18358, 8.45723, 12.6839, 14.8200, 17.3490, 18.9824, 22.5040, ...
# Let me verify by refining the scan.
mp.mp.dps = 30
def find_root_robust(t0, dps=30):
 mp.mp.dps = dps
 s0 = mp.mpc(0.5, t0)
 try:
 return mp.findroot(lambda s: mp.dirichlet(s, chi), s0, solver='muller', tol=mp.mpf(10)**(-dps+5))
 except Exception as e:
 return None

candidates_t = [6.18, 8.46, 12.68, 14.82, 17.35, 18.98, 22.50]
for t0 in candidates_t:
 r = find_root_robust(t0)
 if r:
 print(f"t0={t0}: zero at {r.imag} (Re={r.real})")


t0=6.18: zero at 6.18357819545085391437751730971 (Re=0.5)
t0=8.46: zero at 8.45722917442323072160535286275 (Re=0.5)


t0=12.68: zero at 12.6749464170113557804822991451 (Re=0.5)
t0=14.82: zero at 14.825025570328428251430252174 (Re=0.5)


t0=17.35: zero at 17.3378021068530396909145101424 (Re=0.5)
t0=18.98: zero at 18.9985880416861449287245250119 (Re=0.5)


t0=22.5: zero at 22.4875845830287500250556729093 (Re=0.5)


In [14]:
# All on critical line. Now let's write the full robust pipeline.
# 
# Approach for L-zeros: track Re(L) and Im(L) sign changes simultaneously isn't reliable for 
# complex L. Better approach: use the principle of the argument by scanning |L| with adaptive step.
# Robust method: 
# For each candidate found, also check it's not a duplicate (within tolerance).
# Use density estimate to ensure no zeros missed: at intervals of length dt, expected zero count ~ dt*log(qT)/(2pi)
#
# Let me write this carefully.

import os, time, json
import mpmath as mp
import numpy as np

CHI_4_MOD_5 = [0, 1, mp.mpc(0,1), mp.mpc(0,-1), -1] # chi(0)=0, chi(1)=1, chi(2)=i, chi(3)=-i, chi(4)=-1

def L_chi4(s, chi=CHI_4_MOD_5):
 return mp.dirichlet(s, chi)

# Validation R2: zeta first three zeros
def validate_R2_zeta(dps=50):
 mp.mp.dps = dps
 expected = [mp.mpf('14.134725141'), mp.mpf('21.022039639'), mp.mpf('25.010857580')]
 tol = mp.mpf('1e-8')
 for i, exp in enumerate(expected, 1):
 z = mp.zetazero(i)
 diff = abs(z.imag - exp)
 if diff > tol:
 raise AssertionError(f"R2 failed: zetazero({i})={z.imag}, expected ~{exp}, diff={diff}")
 return True

print("R2 zeta validation:", validate_R2_zeta(50))
print("R2 zeta validation at dps=80:", validate_R2_zeta(80))


R2 zeta validation: True
R2 zeta validation at dps=80: True


In [15]:
# Now design the L-zero finder for chi_4 mod 5.
# Method: scan |L| on a fine grid (step dt), detect local minima below threshold.
# For each minimum, refine via findroot. Track t-values to dedupe.
# Density: avg gap between zeros ~ 2*pi / log(qT/(2*pi)) at height T.
# At T=2000, q=5: gap ~ 2*pi/log(5*2000/(2*pi)) ~ 2*pi/log(1592) ~ 6.28/7.37 ~ 0.85
# At T=20: gap ~ 6.28/log(16) ~ 2.26
# So use adaptive step, or fixed step like 0.1 to be safe.

def find_L_zeros_upto(N, dps=50, scan_step=0.1, refine_tol=None, verbose=False):
 """Find first N positive imaginary parts of zeros of L(s, chi_4 mod 5)."""
 mp.mp.dps = dps
 if refine_tol is None:
 refine_tol = mp.mpf(10) ** (-(dps - 5))
 
 zeros = []
 t = mp.mpf('0.5') # start above 0
 prev_t, prev_val = t, L_chi4(mp.mpc(mp.mpf('0.5'), t))
 prev_mag = abs(prev_val)
 
 # Sliding window for local minima
 window = [] # list of (t, mag, val)
 window.append((prev_t, prev_mag, prev_val))
 
 # Use sign-change in Re(L) and Im(L) as primary indicator combined with magnitude
 while len(zeros) < N:
 t = t + mp.mpf(scan_step)
 val = L_chi4(mp.mpc(mp.mpf('0.5'), t))
 mag = abs(val)
 window.append((float(t), float(mag), val))
 if len(window) > 3:
 window.pop(0)
 # Check local minimum
 if len(window) == 3:
 t0, m0, v0 = window[0]
 t1, m1, v1 = window[1]
 t2, m2, v2 = window[2]
 if m1 < m0 and m1 < m2 and m1 < 0.5:
 # Refine
 try:
 root = mp.findroot(lambda s: L_chi4(s),
 mp.mpc('0.5', str(t1)),
 solver='muller', tol=refine_tol)
 gamma = root.imag
 # Verify on critical line
 if abs(root.real - mp.mpf('0.5')) < mp.mpf('1e-6'):
 # Dedupe
 if not zeros or abs(gamma - zeros[-1]) > mp.mpf('1e-4'):
 zeros.append(gamma)
 if verbose and len(zeros) % 100 == 0:
 print(f" Found {len(zeros)} zeros up to t={gamma}")
 except Exception as e:
 pass
 if t > 10**6: # safety
 break
 return zeros

# Test with small N
t0 = time.time()
zs = find_L_zeros_upto(10, dps=30, verbose=True)
print(f"Found {len(zs)} zeros in {time.time()-t0:.2f}s")
for i, z in enumerate(zs[:10], 1):
 print(f" gamma_{i} = {mp.nstr(z, 15)}")


Found 10 zeros in 3.39s
 gamma_1 = 6.18357819545085
 gamma_2 = 8.45722917442323
 gamma_3 = 12.6749464170114
 gamma_4 = 14.8250255703284
 gamma_5 = 17.337802106853
 gamma_6 = 18.9985880416861
 gamma_7 = 22.4875845830288
 gamma_8 = 24.3652797754023
 gamma_9 = 25.5311868004334
 gamma_10 = 27.9827569356936


In [16]:
# Good. Now let's estimate time for larger N at dps=50.
# At T~2000, density ~ 1 zero per ~0.85 units => 2000 zeros need scanning to T~1700-2000
# With step 0.1 -> ~20000 evaluations. Plus ~2000 refinements at ~0.05s each = 100s.
# Let's verify the scan-evaluation cost.

mp.mp.dps = 50
t0 = time.time()
for tt in np.arange(100, 110, 0.1):
 v = L_chi4(mp.mpc(mp.mpf('0.5'), mp.mpf(str(tt))))
print(f"100 evals near t=100, dps=50: {time.time()-t0:.3f}s")

t0 = time.time()
for tt in np.arange(2000, 2010, 0.1):
 v = L_chi4(mp.mpc(mp.mpf('0.5'), mp.mpf(str(tt))))
print(f"100 evals near t=2000, dps=50: {time.time()-t0:.3f}s")


100 evals near t=100, dps=50: 2.001s


100 evals near t=2000, dps=50: 19.487s


In [17]:
# So at t=2000 each eval ~ 0.2s. 20000 evals at growing t -> avg maybe ~0.1s = 2000s just for scan.
# Plus refinements. Will not finish 2000 L-zeros within budget cleanly.
#
# Strategy refinement: 
# - For ζ, mp.zetazero is the optimal direct algorithm. Use it.
# - For L, use larger scan step (0.3 is safer based on minimum gap), and let findroot handle refinement.
# - The scan is also wasteful; we could use the Riemann-Siegel-like formula but mpmath doesn't 
# provide one for general Dirichlet L-functions directly.
#
# Reduce scope: produce N=2000 zeros for ζ at dps=50 and dps=80; N=2000 for L at dps=50.
# Document feasibility for larger N grids honestly with timing extrapolation.
#
# Actually let me be more aggressive: produce the smallest grid (2000) for both, and projecting
# the larger ones. Time budget per task allocation:
# - Zeta N=2000 dps=50: ~ avg 0.4s/zero * 2000 = 800s
# - Zeta N=2000 dps=80: ~ similar = 800s 
# - L N=2000 dps=50: scan + refine ~ 1500s
# Total ~3000s. Tight but possible. Let me skip L dps=80 due to budget.

# Reconsider time-per-zeta-zero. Let me test sequential indexing 1..200.
mp.mp.dps = 50
t0 = time.time()
zeros_test = []
for n in range(1, 201):
 z = mp.zetazero(n)
 zeros_test.append(z.imag)
t_200 = time.time()-t0
print(f"200 zeta zeros at dps=50: {t_200:.1f}s, avg {t_200/200*1000:.1f}ms/zero")


200 zeta zeros at dps=50: 33.7s, avg 168.3ms/zero


In [18]:
# 168ms/zero at start, growing. For 2000 zeros total at dps=50 maybe ~600-1000s.
# Let me be strategic about scope. The HYPOTHESIS asks whether mpmath CAN reliably do this.
# The answer is YES for any N, but with large compute time. I'll demonstrate:
# - R2 validation passes at both dps levels
# - Functions produce correct zeros (cross-check first few) 
# - Produce N=2000 zeta dps=50 fully
# - Produce N=2000 L dps=50 fully 
# - Smaller proof-of-concept N for dps=80 to demonstrate it works
# - Document time extrapolation
#
# Given the 3600s budget, let me actually first verify the L-finder works for N=200 to validate
# correctness, then commit to N=2000 production for both.

mp.mp.dps = 50
t0 = time.time()
L_zeros_200 = find_L_zeros_upto(200, dps=50, scan_step=0.15)
print(f"200 L-zeros at dps=50: {time.time()-t0:.1f}s")
print(f"Last 5 zeros: {[mp.nstr(z,12) for z in L_zeros_200[-5:]]}")
print(f"Max t: {L_zeros_200[-1]}")


200 L-zeros at dps=50: 564.7s
Last 5 zeros: ['459.426241741', '460.557111261', '461.156899863', '470.068720831', '469.33963528']
Max t: 469.33963527968566758306686602271421618064481885899


In [19]:
# Hmm: 565s for 200 zeros + last few are out of order (459, 460, 461, 470, 469).
# Out-of-order means findroot near t~470 actually converged to a different zero, then findroot at
# next candidate found earlier one back. Need to fix dedup + ordering.
# 
# Also 565s for 200 zeros is way too slow. The scan step 0.15 evaluates at 470/0.15 ~ 3133 points,
# each ~0.05-0.1s = 200-300s alone. Plus refinement.
#
# Critical issue: Plain scan with absolute magnitude check misses some and produces out-of-order.
# 
# Better approach: detect sign changes in Im(L) and Re(L) - for L on the critical line that crosses
# through zero, BOTH real and imaginary parts must change sign at a zero of L. So:
# - When BOTH Re and Im have sign changes between consecutive scan points, refine.
# - This is much more reliable than magnitude minima.
# 
# Actually for a generic complex-valued function L(t), zeros are isolated points in the complex plane.
# On the real line (critical line), L(1/2+it) is a function of real t valued in C. It hits zero when
# both Re(L)=0 and Im(L)=0 simultaneously. Between scan points, both Re and Im typically change sign 
# near a zero.
#
# Even simpler and standard: use the argument of L. The argument winds by 2*pi for each zero crossing.
# Track delta_arg between consecutive scan points and detect when total winding increases by pi.
# 
# But the simplest: use mpmath's "rotated" function that becomes real on critical line.
# For odd primitive character chi mod q, define theta(t) such that 
# Z(t) = (q/pi)^(it/2) * Gamma((1/2+1)/2 + it/2) * L(1/2+it,chi) -- this satisfies a functional eq.
# Actually let me just use the argument-tracking method.

# Use argument tracking:
def find_L_zeros_via_arg(N, dps=50, scan_step=0.2, verbose=False):
 mp.mp.dps = dps
 refine_tol = mp.mpf(10) ** (-(dps - 5))
 zeros = []
 t = mp.mpf('0.5')
 val = L_chi4(mp.mpc('0.5', str(float(t))))
 prev_arg = mp.arg(val)
 prev_t = t
 prev_val = val
 accum_winding = mp.mpf(0)
 
 safety_limit = 50000
 iters = 0
 while len(zeros) < N and iters < safety_limit:
 iters += 1
 t = t + mp.mpf(scan_step)
 val = L_chi4(mp.mpc('0.5', str(float(t))))
 # Argument increment
 d = mp.arg(val) - prev_arg
 # Normalize to (-pi, pi]
 while d > mp.pi:
 d -= 2*mp.pi
 while d <= -mp.pi:
 d += 2*mp.pi
 accum_winding += d
 # A zero occurs when winding decreases by ~pi locally (since L crosses through 0)
 # More robust: check sign change in Re and Im of val
 re_sign_change = (mp.re(prev_val) * mp.re(val)) < 0
 im_sign_change = (mp.im(prev_val) * mp.im(val)) < 0
 mag_min = abs(val) < 2.0 or abs(prev_val) < 2.0
 
 if (re_sign_change or im_sign_change) and mag_min:
 # Refine using findroot in bracket
 t_mid = (prev_t + t) / 2
 try:
 root = mp.findroot(lambda s: L_chi4(s),
 mp.mpc('0.5', str(float(t_mid))),
 solver='muller', tol=refine_tol)
 if abs(mp.re(root) - mp.mpf('0.5')) < mp.mpf('1e-6'):
 g = mp.re(root.imag) if False else root.imag
 # Only accept if within current bracket-ish
 if abs(g - t_mid) < scan_step * 2:
 if not zeros or abs(g - zeros[-1]) > mp.mpf('1e-4'):
 if not zeros or g > zeros[-1]: # enforce order
 zeros.append(g)
 if verbose and len(zeros) % 50 == 0:
 print(f" Found {len(zeros)} zeros, t={float(g):.2f}")
 except Exception:
 pass
 prev_arg = mp.arg(val)
 prev_t, prev_val = t, val
 return zeros

mp.mp.dps = 50
t0 = time.time()
zs = find_L_zeros_via_arg(50, dps=50, scan_step=0.2, verbose=True)
print(f"50 L-zeros in {time.time()-t0:.1f}s")
print(f"First 10: {[mp.nstr(z,12) for z in zs[:10]]}")
print(f"Last few: {[mp.nstr(z,12) for z in zs[-3:]]}")


 Found 50 zeros, t=108.44
50 L-zeros in 58.2s
First 10: ['6.18357819545', '8.45722917442', '12.674946417', '14.8250255703', '17.3378021069', '18.9985880417', '22.487584583', '24.3652797754', '25.5311868004', '27.9827569357']
Last few: ['105.517464526', '106.288695498', '108.437285635']


In [20]:
# 58s for 50 zeros = 1.16s/zero. For 2000 zeros, extrapolating (cost increases with t)...
# Density goes as log(T), so cost per zero roughly constant-to-slowly-increasing.
# 2000 zeros up to T~2000: each L eval at T=2000 takes ~0.2s vs 0.02s at T=100.
# Total: scan steps ~ 2000/0.2 = 10000 evals, total scan time ~ 10000 * avg_eval (~0.05-0.1s) = 500-1000s.
# Plus refinements ~ 2000 * 0.02s = 40s.
# 
# So 2000 L-zeros at dps=50 takes maybe 600-1200s. Acceptable.
# 
# But for ζ via mp.zetazero(n), index 2000 takes ~1s/zero. 2000 zeros maybe 600s.
#
# Total budget so far used: ~700s (mostly from prior experiments). Remaining ~2900s.
# Plan: produce N=2000 zeta at dps=50 and dps=80, N=2000 L at dps=50. Skip larger grids honestly.

# Let me first cleanly generate ζ zeros at dps=50, N=2000

def generate_zeta_zeros(N, dps):
 mp.mp.dps = dps
 zeros = []
 t0 = time.time()
 for n in range(1, N+1):
 z = mp.zetazero(n)
 zeros.append(z.imag)
 return zeros, time.time() - t0

def save_zeros(zeros, path, dps):
 with open(path, 'w') as f:
 f.write(f"# {len(zeros)} zeros (imaginary parts), dps={dps}\n")
 for z in zeros:
 f.write(mp.nstr(z, dps) + "\n")

# R2 validation
def validate_R2(zeros, dps):
 expected = [mp.mpf('14.134725141'), mp.mpf('21.022039639'), mp.mpf('25.010857580')]
 tol = mp.mpf('1e-8')
 for i, exp in enumerate(expected):
 d = abs(zeros[i] - exp)
 if d > tol:
 return False, f"Zero {i+1}: got {zeros[i]}, expected ~{exp}, diff={d}"
 return True, "Pass"

print("Functions defined")


Functions defined


In [21]:
# Generate zeta zeros at dps=50, N=2000
import os
os.makedirs('cached_zeros/zeta', exist_ok=True)
os.makedirs('cached_zeros/dirichlet_chi4_mod5', exist_ok=True)

manifest = {"timing": {}, "files": {}, "validation": {}}

print("Generating zeta zeros: N=2000, dps=50...")
zeta_2k_50, t_zeta_2k_50 = generate_zeta_zeros(2000, 50)
ok, msg = validate_R2(zeta_2k_50, 50)
print(f" R2 validation: {ok} ({msg})")
print(f" Time: {t_zeta_2k_50:.1f}s")
path = 'cached_zeros/zeta/N2000_dps50.txt'
save_zeros(zeta_2k_50, path, 50)
manifest["timing"]["zeta_N2000_dps50"] = t_zeta_2k_50
manifest["files"]["zeta_N2000_dps50"] = path
manifest["validation"]["zeta_N2000_dps50"] = msg
print(f" Saved to {path}")
print(f" Last zero: {mp.nstr(zeta_2k_50[-1], 15)}")


TimeoutError: Code execution timed out after 900.0 seconds

In [22]:
# Hit 900s timeout. Let me check what was generated. Need to recover.
# We'll proceed more incrementally.
import os
import mpmath as mp
import time
import numpy as np

# Re-establish state
CHI_4_MOD_5 = [0, 1, mp.mpc(0,1), mp.mpc(0,-1), -1]
def L_chi4(s, chi=CHI_4_MOD_5):
 return mp.dirichlet(s, chi)

# Check what we have
for root, dirs, files in os.walk('cached_zeros'):
 for f in files:
 p = os.path.join(root, f)
 print(p, os.path.getsize(p))


In [23]:
# Nothing got saved (the assignment hadn't completed). 
# The plan must be redesigned. Issues:
# 1. Cell-level 900s timeout 
# 2. zeta zero generation is slower than estimated past n=1000
# 
# Let me redesign: generate in chunks, save incrementally, and reduce scope honestly.

import mpmath as mp
import time, os

mp.mp.dps = 50

def generate_zeta_chunk(start, end, dps):
 mp.mp.dps = dps
 out = []
 for n in range(start, end+1):
 out.append(mp.zetazero(n).imag)
 return out

# Quick recalibration: time zeros 1..500 at dps=50
t0 = time.time()
chunk1 = generate_zeta_chunk(1, 500, 50)
t_500 = time.time() - t0
print(f"zeta zeros 1..500 at dps=50: {t_500:.1f}s")


zeta zeros 1..500 at dps=50: 134.7s


In [24]:
# 134s for 500 zeros. Extrapolating: zeros grow ~ n^0.5 in cost ~ log scaling.
# 2000 zeros total roughly: 4x the volume but later zeros are slower. Estimate 800-1200s.
# We've used ~1000s of budget so far + 134s = ~1100s. Remaining ~2500s.

# Save chunk1 first
mp.mp.dps = 50
with open('cached_zeros/zeta/N2000_dps50.txt', 'w') as f:
 f.write(f"# zeta zeros (imaginary parts), dps=50, N=2000 (partial: 1..500 written first)\n")
 for z in chunk1:
 f.write(mp.nstr(z, 50) + "\n")
print(f"Wrote 500 zeros, last: {mp.nstr(chunk1[-1], 15)}")


Wrote 500 zeros, last: 811.184358846506


In [25]:
# Continue: zeros 501..1000
t0 = time.time()
chunk2 = generate_zeta_chunk(501, 1000, 50)
t_chunk2 = time.time() - t0
print(f"zeta zeros 501..1000 at dps=50: {t_chunk2:.1f}s")

# Append to file
with open('cached_zeros/zeta/N2000_dps50.txt', 'a') as f:
 for z in chunk2:
 f.write(mp.nstr(z, 50) + "\n")
print(f"Last zero: {mp.nstr(chunk2[-1], 15)}")


zeta zeros 501..1000 at dps=50: 236.6s
Last zero: 1419.422480946


In [26]:
# 236s for 500 zeros (501-1000). Per zero 0.47s. Next 500 (1001-1500) probably ~350s.
# Last 500 (1501-2000) ~450s. Total remaining for zeta_2k_dps50: ~800s.
# We have ~2500s left. Tight. Skip dps=80 production OR do it with smaller N.

t0 = time.time()
chunk3 = generate_zeta_chunk(1001, 1500, 50)
t_chunk3 = time.time() - t0
print(f"zeta zeros 1001..1500: {t_chunk3:.1f}s")

with open('cached_zeros/zeta/N2000_dps50.txt', 'a') as f:
 for z in chunk3:
 f.write(mp.nstr(z, 50) + "\n")
print(f"Last: {mp.nstr(chunk3[-1], 15)}")


zeta zeros 1001..1500: 274.6s
Last: 1980.91004309102


In [27]:
# Tight on time. Get last 500 for zeta and then quickly generate L zeros.
t0 = time.time()
chunk4 = generate_zeta_chunk(1501, 2000, 50)
t_chunk4 = time.time() - t0
print(f"zeta zeros 1501..2000: {t_chunk4:.1f}s")

with open('cached_zeros/zeta/N2000_dps50.txt', 'a') as f:
 for z in chunk4:
 f.write(mp.nstr(z, 50) + "\n")
print(f"Last: {mp.nstr(chunk4[-1], 15)}")

total_zeta_time = 134.7 + 236.6 + 274.6 + t_chunk4
print(f"Total zeta N=2000 dps=50: {total_zeta_time:.1f}s")


zeta zeros 1501..2000: 355.1s
Last: 2515.28648292471
Total zeta N=2000 dps=50: 1001.0s


In [28]:
# Tight! Validate R2 from cached file, also generate zeta at dps=80 with smaller N to demonstrate it works.
import mpmath as mp

# Load and validate
with open('cached_zeros/zeta/N2000_dps50.txt') as f:
 lines = [l.strip() for l in f if l.strip() and not l.startswith('#')]
mp.mp.dps = 50
zeros_loaded = [mp.mpf(l) for l in lines]
print(f"Loaded {len(zeros_loaded)} zeros from cache file")

expected = [mp.mpf('14.134725141'), mp.mpf('21.022039639'), mp.mpf('25.010857580')]
tol = mp.mpf('1e-8')
for i, exp in enumerate(expected):
 d = abs(zeros_loaded[i] - exp)
 print(f" Zero {i+1}: {mp.nstr(zeros_loaded[i],12)}, expected ~{exp}, diff={mp.nstr(d, 3)}, pass={d<tol}")


Loaded 2000 zeros from cache file
 Zero 1: 14.1347251417, expected ~14.134725141, diff=7.35e-10, pass=True
 Zero 2: 21.0220396388, expected ~21.022039639, diff=2.28e-10, pass=True
 Zero 3: 25.0108575801, expected ~25.01085758, diff=1.46e-10, pass=True


In [29]:
# Now quickly: 
# - Small zeta dps=80 demonstration (N=100 to show it works/passes R2)
# - L_chi4 zero list (target N=200 within remaining ~600s budget)

import time
mp.mp.dps = 80

t0 = time.time()
zeta_dps80_small = []
for n in range(1, 101):
 zeta_dps80_small.append(mp.zetazero(n).imag)
t_zeta_80 = time.time()-t0
print(f"zeta N=100 dps=80: {t_zeta_80:.1f}s")

# Save
with open('cached_zeros/zeta/N100_dps80.txt', 'w') as f:
 f.write(f"# zeta zeros (imaginary parts), dps=80, N=100 (demonstration; full grids omitted due to compute budget)\n")
 for z in zeta_dps80_small:
 f.write(mp.nstr(z, 80) + "\n")

# Validate R2 at dps=80
expected = [mp.mpf('14.134725141'), mp.mpf('21.022039639'), mp.mpf('25.010857580')]
for i, exp in enumerate(expected):
 d = abs(zeta_dps80_small[i] - exp)
 print(f" dps=80 Zero {i+1}: diff={mp.nstr(d, 3)}, pass={d<mp.mpf('1e-8')}")


zeta N=100 dps=80: 11.2s
 dps=80 Zero 1: diff=7.35e-10, pass=True
 dps=80 Zero 2: diff=2.28e-10, pass=True
 dps=80 Zero 3: diff=1.46e-10, pass=True
